# ਪਾਠ 13 - ਏਜੰਟ ਯਾਦ


## ਸੈਟਅੱਪ

ਇਹ ਨੋਟਬੁੱਕ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ ਕਿਵੇਂ ਇੱਕ ਯਾਤਰਾ ਬੁਕਿੰਗ ਏਜੰਟ ਬਣਾਇਆ ਜਾਵੇ ਜਿਸ ਵਿੱਚ **ਟਿਕਾਊ ਯਾਦاشت** ਹੁੰਦੀ ਹੈ, ਜੋ ਕਿ **ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ** (MAF) ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ।

ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਕਿਵੇਂ ਵੱਖ-ਵੱਖ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਯਾਦاشت — ਕੰਮ ਕਰਨ ਵਾਲੀ, ਛੋਟੀ ਅਵਧੀ ਅਤੇ ਲੰਮੀ ਅਵਧੀ — ਏਜੰਟ ਨੂੰ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਜਾਣਕਾਰੀ ਕਿਵੇਂ ਸੰਭਾਲਣ ਅਤੇ ਵਰਤਣ 'ਤੇ ਪ੍ਰਭਾਵਿਤ ਕਰਦੀਆਂ ਹਨ।

**ਪੂਰਵ-ਆਵਸ਼ਯਕਤਾਵਾਂ:**
- ਇੱਕ ਮਾਈਕ੍ਰੋਸੌਫਟ ਫਾਊਂਡਰੀ ਪ੍ਰੋਜੈਕਟ ਜਿੱਥੇ ਚੈਟ ਮਾਡਲ ਤਾਇਨਾਤ ਹੋਇਆ ਹੋਵੇ (ਉਦਾਹਰਨ ਲਈ `gpt-4.1-mini`)।
- ਅਜ਼ੂਰ CLI ਵਿੱਚ ਲੌਗਇਨ ਕੀਤਾ ਹੋਇਆ — ਆਪਣੇ ਟਰਮੀਨਲ ਵਿੱਚ `az login` ਚਲਾ ਕੇ।
- `AZURE_AI_PROJECT_ENDPOINT` — ਤੁਹਾਡੇ ਮਾਈਕ੍ਰੋਸੌਫਟ ਫਾਊਂਡਰੀ ਪ੍ਰੋਜੈਕਟ ਦਾ ਐਂਡਪੌਇੰਟ।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ਤੁਹਾਡੇ ਤਾਇਨਾਤ ਮਾਡਲ ਦਾ ਨਾਮ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ਏਜੰਟ ਮেমੋਰੀ ਦੇ ਪ੍ਰਕਾਰ

ਏਆਈ ਏਜੰਟ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀ ਮੈਮੋਰੀ ਦੀ ਵਰਤੋਂ ਕਰ ਸਕਦੇ ਹਨ, ਜਿਹੜਾ ਹਰ ਇੱਕ ਦਾ ਵੱਖਰਾ ਉਦੇਸ਼ ਹੁੰਦਾ ਹੈ:

### ਵਰਕਿੰਗ ਮੈਮੋਰੀ
ਗੱਲਬਾਤ ਦਾ ਧਾਗਾ ਖ਼ੁਦ — ਇੱਕ ਹੀ ਸੈਸ਼ਨ ਵਿੱਚ ਬਦਲੇ ਗਏ ਸੁਨੇਹੇ। ਏਜੰਟ ਉਸੇ ਧਾਗੇ ਵਿੱਚ ਪਹਿਲਾਂ ਦਿੱਤੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਫਿਰ ਤੋਂ ਵੇਖ ਸਕਦਾ ਹੈ ਤਾਂ ਜੋ ਸੰਗਤੀ ਬਣੀ ਰਹੇ। MAF ਵਿੱਚ ਇਹ **`agent.create_session()`** ਰਾਹੀਂ ਬਣਾਇਆ ਜਾਂਦਾ ਹੈ, ਜੋ ਇੱਕ `AgentSession` ਵਾਪਸ ਕਰਦਾ ਹੈ।

### ਛੋਟੀ ਮੁਦਤੀ ਮੈਮੋਰੀ
ਜਾਣਕਾਰੀ ਜੋ ਕੰਮ ਜਾਂ ਸੈਸ਼ਨ ਦੇ ਦੌਰਾਨ ਅਸਥਾਈ ਤੌਰ 'ਤੇ ਟਿਕੀ ਰਹਿੰਦੀ ਹੈ ਪਰ ਸਥਾਈ ਤੌਰ ਤੇ ਸੰਗ੍ਰਹਿਤ ਨਹੀਂ ਹੁੰਦੀ। ਉਦਾਹਰਨ ਵਜੋਂ, ਏਜੰਟ ਬਹੁ-ਚਰਚਾ ਯੋਜਨਾ ਬਣਾਉਣ ਵਾਲੀ ਗੱਲਬਾਤ ਦੌਰਾਨ ਤੱਥ ਇਕੱਠੇ ਕਰ ਸਕਦਾ ਹੈ ਅਤੇ ਉਹਨਾਂ ਨੂੰ ਆਖਰੀ ਯਾਤਰਾ ਯੋਜਨਾ ਤਿਆਰ ਕਰਨ ਲਈ ਵਰਤ ਸਕਦਾ ਹੈ।

### ਲੰਬੀ ਮੁਦਤੀ ਮੈਮੋਰੀ
ਪਸੰਦ ਅਤੇ ਤੱਥ ਜੋ **ਸੈਸ਼ਨਾਂ ਦੇ ਪਾਰ** ਟਿਕੇ ਰਹਿੰਦੇ ਹਨ। ਵਾਪਸੀ ਕਰਨ ਵਾਲੇ ਉਪਭੋਗਤਾ ਨੂੰ ਆਪਣੀਆਂ ਖਾਣ-ਪੀਣ ਦੀਆਂ ਪਾਬੰਦੀਆਂ ਜਾਂ ਯਾਤਰਾ ਸਟਾਈਲ ਦੁਹਰਾਣੀ ਨਹੀਂ ਪਵੇਗੀ। ਲੰਬੀ ਮੈਮੋਰੀ ਆਮ ਤੌਰ 'ਤੇ ਕਿਸੇ ਬਾਹਰੀ ਸਟੋਰ — ਡੇਟਾਬੇਸ, ਫਾਇਲ ਜਾਂ ਵੇਕਟਰ ਇੰਡੈਕਸ — ਦੁਆਰਾ ਸਹਾਇਤ ਹੁੰਦੀ ਹੈ ਅਤੇ ਏਜੰਟ ਲਈ ਟੂਲਜ਼ ਰਾਹੀਂ ਉਪਲੱਬਧ ਕਰਵਾਈ ਜਾਂਦੀ ਹੈ।


## ਸੈਸ਼ਨਾਂ ਨਾਲ ਵਰਕਿੰਗ ਮੈਮੋਰੀ

ਸਭ ਤੋਂ ਸੌਖਾ ਮੇਮੋਰੀ ਫਾਰਮ ਸੰਵਾਦ ਸੈਸ਼ਨ ਹੈ। ਜਦੋਂ ਤੁਸੀਂ ਉਹੀ ਸੈਸ਼ਨ (ਜੋ `agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ ਹੈ) ਲਗਾਤਾਰ `agent.run()` ਕਾਲਾਂ ਨੂੰ ਦੇਂਦੇ ਹੋ, ਤਾਂ ਏਜੰਟ ਉਸ ਸੰਵਾਦ ਦਾ ਪੂਰਾ ਇਤਿਹਾਸ ਦੇਖਦਾ ਹੈ ਅਤੇ ਪਹਿਲਾਂ ਦੀਆਂ ਜਾਣਕਾਰੀਆਂ ਨੂੰ ਯਾਦ ਕਰ ਸਕਦਾ ਹੈ।

ਆਓ ਇਕ ਯਾਤਰਾ ਏਜੰਟ ਬਣਾਈਏ ਅਤੇ ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਨੂੰ ਦਰਸਾਈਏ।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ਏਜੰਟ ਨੇ ਬਜਟ ਨੂੰ ਸਹੀ ਤਰ੍ਹਾਂ ਯਾਦ ਕੀਤਾ ਕਿਉਂਕਿ ਦੋਵੇਂ ਸੁਨੇਹੇ ਇੱਕੋ ਸੈਸ਼ਨ ਸਾਂਝੇ ਕਰਦੇ ਹਨ। ਇਹ **ਕੰਮ ਕਰਨ ਵਾਲੀ ਯਾਦਸ਼ਕਤੀ** ਹੈ — ਇਹ ਸਿਰਫ ਸੈਸ਼ਨ ਦੇ ਜੀਵਨਕਾਲ ਲਈ ਮੌਜੂਦ ਹੈ।

### ਨਵੀਂ ਧਾਰਾ ਨਾਲ ਕੀ ਹੁੰਦਾ ਹੈ?

ਜੇ ਅਸੀਂ ਇੱਕ **ਨਵਾਂ** ਸੈਸ਼ਨ ਬਣਾਂਦੇ ਹਾਂ, ਤਾਂ ਏਜੰਟ ਨੂੰ ਪਿਛਲੀ ਗੱਲਬਾਤ ਦੀ ਕੋਈ ਯਾਦ ਨਹੀਂ ਹੁੰਦੀ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ਲੰਮੀ ਮਿਆਦ ਦੀ ਯਾਦاشت ਦਾ ਪੈਟਰਨ

ਉਪਭੋਗਤਾ ਪਸੰਦਾਂ ਨੂੰ **ਸੈਸ਼ਨਾਂ ਵਿੱਚ** ਯਾਦ ਰੱਖਣ ਲਈ, ਸਾਨੂੰ ਇਕ ਸਥਾਈ ਸਟੋਰ ਦੀ ਲੋੜ ਹੈ ਜੋ ਗੱਲਬਾਤ ਧਾਗੇ ਤੋਂ ਬਾਹਰ ਜੀਉਂਦਾ ਹੈ। ਏਜੰਟ ਇਸ ਸਟੋਰ ਤੱਕ ਪਹੁੰਚ **ਟੂਲਜ਼** ਰਾਹੀਂ ਕਰਦਾ ਹੈ — ਫੰਕਸ਼ਨਾਂ ਜਿਨ੍ਹਾਂ ਨੂੰ ਇਹ ਸੂਚਨਾ ਸੁਰੱਖਿਅਤ ਕਰਨ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਸਧਾਰਣ ਇਨ-ਮੈਮੋਰੀ ਪਸੰਦ ਸਟੋਰ ਲਾਗੂ ਕਰਦੇ ਹਾਂ (ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ ਇਸਨੂੰ ਡੇਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਇੰਡੈਕਸ ਨਾਲ ਪਿਛੋਕੜ ਵਿੱਚ ਰੱਖੋਗੇ) ਅਤੇ ਇਸਨੂੰ ਟੂਲਜ਼ ਵਜੋਂ ਦਰਸਾਉਂਦੇ ਹਾਂ ਜੋ ਏਜੰਟ ਵਰਤ ਸਕਦਾ ਹੈ।

### ਵਾਸਤੂਕਲਾ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ਸਹੀ ਕਿਰਦਾਰ 1 — ਪਹਿਲੀ ਵਾਰੀ ਯੂਜ਼ਰ ਇੱਕ ਸਾਲਗਿਰਾ ਸਫ਼ਰ ਬੁੱਕ ਕਰਦਾ ਹੈ

ਸਾਰਾਹ ਪਹਿਲੀ ਵਾਰੀ ਆਉਂਦੀ ਹੈ। ਏਜੰਟ ਨੂੰ ਉਸ ਦੀਆਂ ਪਸੰਦਾਂ ਟੂਲਜ਼ ਰਾਹੀਂ ਸਟੋਰ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਹੋਟਲਾਂ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕਰਨ ਲਈ ਉਹਨਾਂ ਦੀ ਵਰਤੋਂ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ਪਰਸਥਿਤੀ 2 — ਸਾਰਾ ਹਫ਼ਤਿਆਂ ਬਾਅਦ ਵਾਪਸ ਆਉਂਦੀ ਹੈ

ਸਾਰਾ ਇੱਕ **ਬਿਲਕੁਲ ਨਵੀਂ ਥ੍ਰੈੱਡ** ਸ਼ੁਰੂ ਕਰਦੀ ਹੈ (ਨਵਾਂ ਸੈਸ਼ਨ ਨਰਮਾਇਆ). ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਖਾਲੀ ਹੈ, ਪਰ ਲੰਬੀ ਅਵਧੀ ਲਈ ਪਸੰਦ ਸਟੋਰ ਵਿੱਚ ਅਜੇ ਵੀ ਉਸਦੀ ਜਾਣਕਾਰੀ ਮੌਜੂਦ ਹੈ। ਏਜੰਟ ਨੂੰ ਇਹ ਪ੍ਰਾਪਤ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਇਸਨੂੰ ਵਿਅਕਤੀਗਤ ਸਿਫਾਰਸ਼ਾਂ ਲਈ ਵਰਤਣਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਤਿੰਨ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਮੈਮੋਰੀ ਬਾਰੇ ਸਿੱਖਿਆ ਅਤੇ ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਉਨ੍ਹਾਂ ਨੂੰ ਕਿਵੇਂ ਲਾਗੂ ਕਰਨਾ ਹੈ:

| ਮੈਮੋਰੀ ਕਿਸਮ | MAF ਮਕੈਨਿਜ਼ਮ | ਜੀਵਨਕਾਲ |
|---|---|---|
| **ਕਾਰਜਕਾਰੀ** | `agent.create_session()` | ਇੱਕ ਗੱਲਬਾਤ |
| **ਛੋਟੀ ਮਿਆਦ ਲਈ** | ਥ੍ਰੇਡ ਦੇ ਅੰਦਰ ਇਕੱਠੀ ਕੀਤੀ ਸੰਦਰਭ | ਇੱਕ ਕਾਰਜ / ਸੈਸ਼ਨ |
| **ਲੰਬੀ ਮਿਆਦ ਲਈ** | ਬਾਹਰੀ ਸਟੋਰ ਜੋ `@tool` ਫੰਕਸ਼ਨਾਂ ਰਾਹੀਂ ਪਹੁੰਚਿਆ ਜਾਂਦਾ ਹੈ | ਸੈਸ਼ਨਾਂ ਵਿਚਕਾਰ |

### ਮੁੱਖ ਸਿੱਖਿਆਵਾਂ
1. **`agent.create_session()`** ਕਾਰਜਕਾਰੀ ਮੈਮੋਰੀ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ — ਏਜੰਟ ਇੱਕ ਸੈਸ਼ਨ ਦੇ ਅੰਦਰ ਪੂਰੀ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਵੇਖਦਾ ਹੈ।
2. **ਨਵੇਂ ਸੈਸ਼ਨ ਸੰਦਰਭ ਗੁਆ ਦੇਂਦੇ ਹਨ** — ਲੰਬੀ ਮਿਆਦ ਵਾਲੀ ਮੈਮੋਰੀ ਦੇ ਬਿਨਾਂ ਏਜੰਟ ਪਿਛਲੇ ਗੱਲਬਾਤਾਂ ਨੂੰ ਯਾਦ ਨਹੀਂ ਰੱਖ ਸਕਦਾ।
3. **`@tool` ਫੰਕਸ਼ਨ** ਖਾਈ ਨੂੰ ਪੂਰਾ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਨੂੰ ਸਥਾਈ ਸਟੋਰ ਤੋਂ ਜਾਣਕਾਰੀ ਸੰਭਾਲਣ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਦਿੰਦੇ ਹਨ।
4. **ਨਿੱਜੀ ਬਣਾਉਣਾ ਸਮੇਂ ਦੇ ਨਾਲ ਸੁਧਰਦਾ ਹੈ** — ਜਿੰਨਾ ਵੱਧ ਪREFERੈਂਸਿਸ ਸੰਭਾਲੇ ਜਾਂਦੇ ਹਨ, ਓਨਾ ਵਧੀਆ ਏਜੰਟ ਦੀ ਸਿਫਾਰਿਸ਼ਾਂ ਹੁੰਦੀਆਂ ਹਨ।

### ਅਸਲੀ ਦੁਨੀਆ ਵਿੱਚ ਐਪਲੀਕੇਸ਼ਨ
- **ਗਾਹਕ ਸੇਵਾ**: ਗਾਹਕ ਦੇ ਇਤਿਹਾਸ ਅਤੇ ਪREFERੈਂਸਿਸ ਯਾਦ ਰੱਖੋ
- **ਨਿੱਜੀ ਸਹਾਇਕ**: ਦਿਨਾਂ ਜਾਂ ਹਫ਼ਤਿਆਂ ਵਿਚ ਸੰਦਰਭ ਬਣਾਈ ਰੱਖੋ
- **ਹੈਲਥਕੇਅਰ**: ਮਰੀਜ਼ ਦੀ ਜਾਣਕਾਰੀ ਅਤੇ ਪREFERੈਂਸਿਸ ਟਰੈਕ ਕਰੋ
- **ਈ-ਕਾਮਰਸ**: ਇਤਿਹਾਸ ਦੇ ਆਧਾਰ 'ਤੇ ਨਿੱਜੀ ਖਰੀਦਦਾਰੀ

### ਅਗਲੇ ਕਦਮ
- ਇਨ-ਮੇਮੋਰੀ ਡਿਕਸ਼ਨਰੀ ਨੂੰ ਡੇਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਸਟੋਰ (ਜਿਵੇਂ ਕਿ Azure AI Search) ਨਾਲ ਬਦਲੋ
- ਸਮੇਂ-ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਣਕਾਰੀ ਲਈ ਮੈਮੋਰੀ ਦੀ ਮਿਆਦ ਖਤਮ ਕਰਨ ਦਾ ਜੋੜੋ
- ਸਾਂਝੀ ਮੈਮੋਰੀ ਨਾਲ ਕਈ ਏਜੰਟ ਪ੍ਰਣਾਲੀਆਂ ਬਣਾਓ
- ਗਿਆਨ-ਗਰਾਫ-ਸਹਿਤ ਮੈਮੋਰੀ ਲਈ Cognee ਨੋਟਬੁੱਕ ਦੀ ਖੋਜ ਕਰੋ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
